# 25 - Agent Identity & Authorization

## Scenario: Role-Based Access Control (RBAC)

If you give an agent a `delete_database` tool, how do you stop a standard user from tricking the agent into deleting the database? 

The agent must adopt the **Identity** of the user who invoked it. When the LLM requests a tool call, the underlying Python function must check the User's RBAC role, NOT the Agent's role.

In this notebook, we implement RBAC for Northstar Support.

In [1]:
# 1. Initialization and Mock Fallback
import os
import sys

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("✅ Using real OpenAI API.")
else:
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    sys.path.append(os.path.abspath("../../.."))
    try:
        from awsome_agents.mock_openai import MockOpenAI
        client = MockOpenAI()
    except ImportError:
        print("Failed to import MockOpenAI. Ensure you are running from the repository root.")

⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. Defining RBAC Tools

In [2]:
# Simulated User Database
USER_ROLES = {
    "alice_admin": "admin",
    "bob_user": "customer"
}

def delete_user_account(target_user: str, executing_user: str) -> str:
    """Tool: Deletes an account. Requires ADMIN privileges."""
    role = USER_ROLES.get(executing_user, "guest")
    print(f"  🔒 [RBAC Check] User '{executing_user}' is role '{role}'")
    
    if role != "admin":
        return f"ERROR: Permission Denied. '{executing_user}' is not an admin."
    
    print(f"  🗑️ [Action] Deleted account {target_user} successfully.")
    return "Account deleted."


## 2. Agent Context Execution

In [3]:
def execute_agent_request(query: str, current_user: str):
    print(f"\n📩 Request from {current_user}: '{query}'")
    print("🧠 [Agent] Formulating plan...")
    
    # The agent decides to call the delete tool.
    # CRITICAL: We pass the `current_user` into the tool behind the scenes!
    result = delete_user_account(target_user="charlie_test", executing_user=current_user)
    print(f"🤖 [Agent Response] {result}")

# 1. Bob tries to delete an account (Should fail)
execute_agent_request("Please delete charlie's account immediately.", current_user="bob_user")

# 2. Alice tries to delete an account (Should succeed)
execute_agent_request("Delete charlie's account.", current_user="alice_admin")



📩 Request from bob_user: 'Please delete charlie's account immediately.'
🧠 [Agent] Formulating plan...
  🔒 [RBAC Check] User 'bob_user' is role 'customer'
🤖 [Agent Response] ERROR: Permission Denied. 'bob_user' is not an admin.

📩 Request from alice_admin: 'Delete charlie's account.'
🧠 [Agent] Formulating plan...
  🔒 [RBAC Check] User 'alice_admin' is role 'admin'
  🗑️ [Action] Deleted account charlie_test successfully.
🤖 [Agent Response] Account deleted.


## Checkpoint

**1. Why is passing the `executing_user` to the tool critical for security?**
- A) To make the prompt longer.
- B) Because the LLM cannot be trusted to enforce authorization. The underlying code must enforce RBAC based on the identity of the human driving the session.
- C) So the LLM can email the user.
- D) To bypass OAuth.
